# Lekcja 2.4 — eksperymenty: heurystyki w magazynie

Uruchamiaj komórki po kolei. Przed każdym eksperymentem zapisz swoje przewidywanie w komórce „Twoje przewidywanie” (kliknij ją dwukrotnie, żeby edytować), a dopiero potem uruchom kod.

Pierwsza komórka wybiera implementację:

- `"mine"` używa Twojego `task.py`; najpierw rozwiąż część B zadania;
- `"reference"` używa implementacji wzorcowej kursu. Wpisz `IMPLEMENTATION = "reference"`, jeśli chcesz uruchomić notebook przed skończeniem zadania.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    from task import a_star, greedy_best_first, manhattan
else:
    from introai.search.reference import a_star, greedy_best_first
    from introai.world.reference import manhattan

Funkcja `run` uruchamia jedno przeszukiwanie na mapie i zwraca węzeł celu oraz listę rozwiniętych węzłów. Przeszukiwania z heurystyką dostają funkcję `h(cell)`, a funkcja `to_dock(warehouse, distance)` buduje taką funkcję z odległości od rampy. UCS pochodzi zawsze z implementacji wzorcowej kursu (lekcja 2.3). Na rysunkach niebieskie pola to pola rozwinięte, pomarańczowe to trasa, a zielone, kreskowane pola to strefy wolnego ruchu.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from introai.search.reference import ucs
from introai.trace import TraceLimitExceeded, record
from introai.world.plot import plot_map
from introai.world.warehouse import RouteProblem, WarehouseMap, render_text

MAPS = Path.cwd().parents[2] / "data" / "maps"


def to_dock(warehouse, distance):
    """A heuristic: the given distance from a cell to the dock."""
    return lambda cell: distance(cell, warehouse.dock)


def run(search, warehouse, h=None, limit=100_000):
    """Run one search on a map; return the goal node and the expanded nodes."""
    problem = RouteProblem(warehouse)
    try:
        with record(limit=limit, kinds={"expand"}) as trace:
            node = search(problem) if h is None else search(problem, h)
    except TraceLimitExceeded:
        raise RuntimeError(
            f"{search.__name__} expanded more than {limit} nodes - check task.py "
            '(or set IMPLEMENTATION = "reference" in the first cell)'
        ) from None
    return node, [event.data["node"] for event in trace.of_kind("expand")]


def summary(label, node, expanded):
    if node is None:
        print(f"{label}: no route, expanded {len(expanded)}")
    else:
        print(
            f"{label}: {len(node.solution())} steps, cost {node.path_cost:g}, "
            f"expanded {len(expanded)}"
        )

## Eksperyment 1: UCS, greedy i A* w zatłoczonym magazynie

Ta sama mapa co w lekcji 2.3. Greedy i A\* dostają odległość Manhattan do rampy.

In [ ]:
congested = WarehouseMap.from_file(MAPS / "congested.txt")
print(render_text(congested))

**Twoje przewidywanie.** Którędy pojedzie każdy z trzech algorytmów? Który rozwinie najwięcej pól, a który najmniej? Czy A\* zajrzy do magazynu po lewej stronie?

*Moje przewidywanie:* …

In [ ]:
h = to_dock(congested, manhattan)
runs = [
    ("UCS", run(ucs, congested)),
    ("greedy best-first", run(greedy_best_first, congested, h)),
    ("A*", run(a_star, congested, h)),
]
fig, axes = plt.subplots(3, 1, figsize=(10, 9))
for ax, (label, (node, expanded)) in zip(axes, runs):
    summary(label, node, expanded)
    path = node.states() if node else []
    plot_map(congested, path=path, expanded=[n.state for n in expanded], title=label, ax=ax)
fig.tight_layout()

**Do przemyślenia.** Porównaj wynik ze swoim przewidywaniem. Dlaczego greedy wjechał w korytarz, choć to najdroższa trasa? Które pola w magazynie po lewej rozwinął A\* i dlaczego właśnie te? Policz dla jednego z nich *f* = *g* + *h* i porównaj z kosztem trasy.

## Eksperyment 2: słabsza i silniejsza heurystyka

Porównujemy A\* z pięcioma heurystykami: *h* = 0, odległością euklidesową (w linii prostej), odległością Manhattan oraz odległością Manhattan pomnożoną przez 2 i przez 3.

**Twoje przewidywanie.** Które z tych heurystyk są dopuszczalne? Jak zmieni się liczba rozwinięć i koszt trasy, gdy heurystyka rośnie?

*Moje przewidywanie:* …

In [ ]:
def euclidean(cell, goal):
    return ((cell[0] - goal[0]) ** 2 + (cell[1] - goal[1]) ** 2) ** 0.5


heuristics = {
    "h = 0": lambda cell: 0,
    "Euclidean": to_dock(congested, euclidean),
    "Manhattan": to_dock(congested, manhattan),
    "2 x Manhattan": lambda cell: 2 * manhattan(cell, congested.dock),
    "3 x Manhattan": lambda cell: 3 * manhattan(cell, congested.dock),
}
labels, expansions, costs = [], [], []
for label, h in heuristics.items():
    node, expanded = run(a_star, congested, h)
    summary(f"A* with {label}", node, expanded)
    labels.append(label)
    expansions.append(len(expanded))
    costs.append(node.path_cost)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, values, title in [
    (axes[0], expansions, "Cells expanded by A*"),
    (axes[1], costs, "Cost of the route A* finds"),
]:
    ax.bar(labels, values, color="#2a78d6", width=0.6)
    ax.set_title(title, loc="left", color="#0b0b0b")
    ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c3c2b7")
    ax.tick_params(colors="#52514e")
    ax.tick_params(axis="x", labelrotation=20)
fig.tight_layout()

**Do przemyślenia.** Które z tych heurystyk są dopuszczalne? Przy której heurystyce A\* po raz pierwszy zwraca droższą trasę? Czy podwojona odległość Manhattan jest dopuszczalna, skoro na tej mapie A\* z nią nadal znalazł najtańszą trasę? Odległość euklidesowa rozwinęła tu tyle samo pól co Manhattan. W eksperymencie 3 sprawdzisz, czy tak jest zawsze.

## Eksperyment 3: alejki i otwarta podłoga

Na tych mapach nie ma stref wolnego ruchu, więc odległość Manhattan bywa bardzo bliska prawdziwemu kosztowi.

**Twoje przewidywanie.** Ile razy mniej pól rozwinie A\* z odległością Manhattan niż UCS na mapie z alejkami, a ile razy na otwartej podłodze? Czy A\* z odległością euklidesową rozwinie tyle samo pól?

*Moje przewidywanie:* …

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 7))
for row, name in enumerate(["aisles", "open_floor"]):
    warehouse = WarehouseMap.from_file(MAPS / f"{name}.txt")
    node, expanded = run(a_star, warehouse, to_dock(warehouse, euclidean))
    summary(f"{name}, A* with Euclidean", node, expanded)
    for col, (label, search, h) in enumerate(
        [("UCS", ucs, None), ("A* with Manhattan", a_star, to_dock(warehouse, manhattan))]
    ):
        node, expanded = run(search, warehouse, h)
        summary(f"{name}, {label}", node, expanded)
        path = node.states() if node else []
        expanded_cells = [n.state for n in expanded]
        title = f"{name}: {label}"
        plot_map(warehouse, path=path, expanded=expanded_cells, title=title, ax=axes[row][col])
fig.tight_layout()

**Do przemyślenia.** Na której mapie heurystyka pomaga bardziej i dlaczego? Kiedy odległość Manhattan jest równa prawdziwemu kosztowi trasy, a kiedy go zaniża? Porównaj odległość euklidesową z odległością Manhattan dla jednego pola: która jest większa? Jak to się przekłada na liczbę rozwinięć?

## Podsumowanie

Wróć do pytań kontrolnych w lekcji. Po własnej próbie możesz porównać swoje wyniki i wnioski z odpowiedziami w `solutions/02_search/04_heuristics_and_a_star/answers.pl.md`.